# 24 — Error Handling and Fallbacks

Retry logic, model fallbacks, and graceful degradation.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
import time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Example 1: Built-in Retry

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_retries=3, request_timeout=30)
chain = ChatPromptTemplate.from_template("What is {topic}? Answer in one sentence.") | llm | StrOutputParser()

result = chain.invoke({"topic": "the Pythagorean theorem"})
print(f"Result: {result}\nConfig: max_retries=3, timeout=30s")

## Example 2: Model Fallback Chain

In [ ]:
primary = ChatOpenAI(model="gpt-4o-mini", temperature=0)
fallback = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
prompt = ChatPromptTemplate.from_template("Explain {concept} clearly in 2 sentences.")

primary_chain = prompt | primary | StrOutputParser()
fallback_chain = prompt | fallback | StrOutputParser()
chain_with_fallback = primary_chain.with_fallbacks([fallback_chain])

for concept in ["quantum entanglement", "blockchain consensus"]:
    print(f"{concept}: {chain_with_fallback.invoke({'concept': concept})}\n")

## Example 3: Graceful Degradation

In [ ]:
def safe_invoke(chain, inputs, default="I'm unable to answer right now."):
    try:
        return chain.invoke(inputs)
    except Exception as e:
        print(f"  [Error: {type(e).__name__}: {e}]")
        return default

print(f"Normal: {safe_invoke(chain, {'topic': 'gravity'})}")

bad_chain = ChatPromptTemplate.from_template("What is {topic}?") | ChatOpenAI(model="gpt-nonexistent", max_retries=0) | StrOutputParser()
print(f"Failed: {safe_invoke(bad_chain, {'topic': 'gravity'}, default='[Fallback] Gravity is a fundamental force.')}")

## Example 4: Custom Retry with Backoff

In [ ]:
def with_retry_backoff(fn, max_retries=3, base_delay=1.0):
    def wrapper(*args, **kwargs):
        for attempt in range(max_retries + 1):
            try:
                return fn(*args, **kwargs)
            except Exception as e:
                if attempt == max_retries: raise
                delay = base_delay * (2 ** attempt)
                print(f"  [Retry {attempt+1}] waiting {delay:.1f}s...")
                time.sleep(delay)
    return wrapper

chain = ChatPromptTemplate.from_template("Define '{term}' in one sentence.") | llm | StrOutputParser()

@with_retry_backoff
def resilient_call(term):
    return chain.invoke({"term": term})

for term in ["idempotency", "eventual consistency"]:
    print(f"{term}: {resilient_call(term)}\n")